In [1]:
pip install nltk


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from typing import List, Set, Dict, Any

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [3]:
def get_written_dates_and_any_text_file_locations_to_dataframe(root_folder):
    print(f"--- Starting Scan for metadata.json and any associated non-JSON text files ---")
    print(f"Scanning root folder: {os.path.abspath(root_folder)}\n")

    if not os.path.exists(root_folder):
        print(f"Error: The root folder '{root_folder}' does not exist at '{os.path.abspath(root_folder)}'.")
        return pd.DataFrame()

    data_for_df = []
    
    for dirpath, dirnames, filenames in os.walk(root_folder):
        metadata_filepath_in_dir = None
        text_file_path_in_dir = None
        
        non_json_files = []

        # First, identify the metadata.json and any non-JSON files in the current directory
        for filename in filenames:
            if filename.lower() == "metadata.json":
                metadata_filepath_in_dir = os.path.join(dirpath, filename)
            elif filename.lower().endswith(".txt"): # Consider anything that's not JSON as a potential text file
                non_json_files.append(os.path.join(dirpath, filename))

        # Check conditions for a valid pair
        if metadata_filepath_in_dir:
            if len(non_json_files) == 1:
                # Exactly one non-JSON file found, assume it's the text file
                text_file_path_in_dir = non_json_files[0]
                
                print(f"Processing directory: {os.path.abspath(dirpath)}")
                print(f"  Found metadata.json: {metadata_filepath_in_dir}")
                print(f"  Found associated text file: {text_file_path_in_dir}")

                extracted_written_date = None
                try:
                    with open(metadata_filepath_in_dir, 'r', encoding='utf-8') as f:
                        metadata = json.load(f)
                        if 'written_date' in metadata:
                            extracted_written_date = metadata['written_date']
                            print(f"  Extracted 'written_date': '{extracted_written_date}'")
                        else:
                            print(f"  Warning: 'written_date' key NOT found in {metadata_filepath_in_dir}. Setting to None for this entry.")
                except json.JSONDecodeError as e:
                    print(f"  Error: Could not parse JSON from {metadata_filepath_in_dir}: {e}. Setting 'written_date' to None.")
                except Exception as e:
                    print(f"  An unexpected error occurred reading {metadata_filepath_in_dir}: {e}. Setting 'written_date' to None.")
                
                # Add this pair to our DataFrame data
                data_for_df.append({
                    'written_date': extracted_written_date,
                    'text_file_location': os.path.abspath(text_file_path_in_dir)
                })
                
            elif len(non_json_files) == 0:
                print(f"  Note: Found metadata.json in {os.path.abspath(dirpath)} but NO non-JSON text file found. Skipping this directory.")
            else: # len(non_json_files) > 1
                print(f"  Warning: Found metadata.json in {os.path.abspath(dirpath)} but MULTIPLE non-JSON files ({non_json_files}). Cannot determine which is the text file. Skipping this directory.")
        # No else for when metadata.json is not found at all, to keep output cleaner

    print(f"\n--- Scan Complete ---")
    print(f"Total valid pairs (metadata.json + single non-JSON text file): {len(data_for_df)}")
    
    df = pd.DataFrame(data_for_df)

    if df.empty:
        print("No valid pairs of metadata.json and single non-JSON text file found to create a DataFrame.")
    else:
        print("\n--- Resulting DataFrame Head ---")
        print(df.head())
        print(f"\nDataFrame has {len(df)} rows and {len(df.columns)} columns.")
    
    return df

In [4]:
root_directory_to_scan = "data/OCR_Final"

list_of_file_locations = get_written_dates_and_any_text_file_locations_to_dataframe(root_directory_to_scan)

--- Starting Scan for metadata.json and any associated non-JSON text files ---
Scanning root folder: /Users/nevidujayatilleke/Documents/MSC - Research/OCR of Corpus/SiDiaC-v.2.0/data/OCR_Final

Processing directory: /Users/nevidujayatilleke/Documents/MSC - Research/OCR of Corpus/SiDiaC-v.2.0/data/OCR_Final/ධර්ම ප්‍රදීපිකාව හෙවත් මහාබෝධිවංශ පරිකථාව
  Found metadata.json: data/OCR_Final/ධර්ම ප්‍රදීපිකාව හෙවත් මහාබෝධිවංශ පරිකථාව/metadata.json
  Found associated text file: data/OCR_Final/ධර්ම ප්‍රදීපිකාව හෙවත් මහාබෝධිවංශ පරිකථාව/ධර්ම ප්‍රදීපිකාව හෙවත් මහාබෝධිවංශ පරිකථාව.txt
  Extracted 'written_date': '1187 - 1225'
Processing directory: /Users/nevidujayatilleke/Documents/MSC - Research/OCR of Corpus/SiDiaC-v.2.0/data/OCR_Final/දුක්ඛ සත්‍ය වර්ණනා
  Found metadata.json: data/OCR_Final/දුක්ඛ සත්‍ය වර්ණනා/metadata.json
  Found associated text file: data/OCR_Final/දුක්ඛ සත්‍ය වර්ණනා/දුක්ඛ සත්‍ය වර්ණනා.txt
  Extracted 'written_date': ''
Processing directory: /Users/nevidujayatilleke/Documents/MS

In [5]:
list_of_file_locations.head()

,written_date,text_file_location
0,1187 - 1225,/Users/nevidujayatilleke/Documents/MSC - Resea...
1,,/Users/nevidujayatilleke/Documents/MSC - Resea...
2,1832 - 1907,/Users/nevidujayatilleke/Documents/MSC - Resea...
3,1901 - 1938,/Users/nevidujayatilleke/Documents/MSC - Resea...
4,1944,/Users/nevidujayatilleke/Documents/MSC - Resea...


In [6]:
def clean_words(data, path = True):
    try:
        if path:
            with open(data, 'r', encoding='utf-8') as file:
                text = file.read()
        else:
            text = data
        # Pattern: Split by whitespace OR find and keep the <psi> token
        tokens = re.split(r'(\s+|<psi>)', text)
        
        # Filter out empty strings and pure whitespace left by re.split()
        tokens = [t.strip() for t in tokens if t.strip()]  # FIX: keep '<psi>' so the merge loop below can see it

        # 2. Handle the <psi> token by concatenating adjacent words
        processed_tokens = []
        i = 0
        while i < len(tokens):
            if tokens[i] == '<psi>':
                # Check for preceding and succeeding words
                if i > 0 and i < len(tokens) - 1:
                    # Concatenate the left word, right word, and replace the <psi>
                    new_word = tokens[i-1] + tokens[i+1]
                    # Replace the previous token (which was added in the previous loop iteration)
                    processed_tokens[-1] = new_word
                    i += 2  # Skip <psi> and the concatenated right word
                elif i > 0:
                    # <psi> is at the end, just remove it.
                    i += 1
                elif i < len(tokens) - 1:
                    # <psi> is at the start, just remove it.
                    i += 1
                else:
                    # <psi> is the only token, just remove it.
                    i += 1
            else:
                processed_tokens.append(tokens[i])
                i += 1

        # Re-join the processed tokens into a single string for cleaning
        processed_text = ' '.join(processed_tokens)

        # 3. Remove non-Sinhala characters (keeping Sinhala Unicode range and whitespace)
        cleaned = re.sub(r'[^\u0D80-\u0DFF\s]', '', processed_text)

        # 4. Tokenize by whitespace and count
        words = cleaned.split()
        return words

    except Exception as e:
        print(f"Error reading {path}: {e}")
        return 0

In [7]:
all_text = ""
for path in list_of_file_locations['text_file_location']:
    try:
        with open(path, 'r', encoding='utf-8') as file:
            all_text += file.read()
    except Exception as e:
        print(f"Error reading {path}: {e}")

In [8]:
cleaned_text = clean_words(all_text, path = False)

In [9]:
import nltk
import string
from collections import defaultdict, Counter
from typing import List, Dict, Tuple

In [10]:
def find_context_cooccurrences(tokens: List[str], window: int) -> Dict[str, Counter]:
    """
    Finds all words co-occurring within a +/- window of a target word.

    Args:
        tokens: The list of pre-processed words.
        window: The +/- context window size.

    Returns:
        A dictionary where keys are target words and values are Counter 
        objects of their co-occurring neighbors.
    """
    # Use defaultdict to automatically create a Counter object for new target words
    cooccurrence_map = defaultdict(Counter)
    
    # Iterate through every token in the text, treating it as the target word
    for i, target_word in enumerate(tokens):
        # Define the start and end indices for the window
        start_index = max(0, i - window)
        end_index = min(len(tokens), i + window + 1) # +1 because slice end is exclusive
        
        # Extract the context window slice
        context_window = tokens[start_index:end_index]
        
        # Add all words in the window to the target word's Counter
        for neighbor in context_window:
            # We skip the target word itself to only count its neighbors
            if neighbor != target_word:
                cooccurrence_map[target_word][neighbor] += 1
                
    return cooccurrence_map

In [11]:
WINDOW_SIZE = 10 

cooccurrence_results = find_context_cooccurrences(cleaned_text, WINDOW_SIZE)

In [12]:
cooccurrence_results['මිතුර']

Counter({'තොප': 12,
         'කල': 9,
         'අප': 5,
         'වලා': 5,
         'සිටි': 4,
         'තැනටත්': 4,
         'කෙරෙති': 4,
         'වෙත': 4,
         'ඉන්': 4,
         'සිත': 4,
         'යන': 4,
         'සඳ': 4,
         'සැටි': 3,
         'දිනති': 3,
         'අහස': 3,
         'යස': 3,
         'පවසමි': 3,
         'කෙලෙස': 3,
         'මන': 3,
         'මියුරු': 3,
         'වන': 3,
         'සිරි': 3,
         'පෙර': 3,
         'රැගෙණ': 2,
         'වැඩෙයි': 2,
         'අන්වය': 2,
         'ද': 2,
         'හිරු': 2,
         'රුති': 2,
         'බැති': 2,
         'වැඩෙ': 2,
         'යි': 2,
         'අදහස': 2,
         'පසුකර': 2,
         'මරකත': 2,
         'මිණි': 2,
         'ගොදුරු': 2,
         'පිපි': 2,
         'හිස': 2,
         'ගමන්': 2,
         'ලක්': 2,
         'වත්': 2,
         'සසර': 2,
         'සොඳ': 2,
         'ඉත': 2,
         'දැක': 2,
         'යනු': 2,
         'පවර': 2,
         'සිතින': 2,
         'මැන': 2,
         'යුතු': 2,

In [13]:
def print_top_cooccurrences(results: Dict[str, Counter], target_word: str, top_n: int = 5):
    """Prints the top co-occurring words for a specific target word."""
    if target_word in results:
        print(f"\n--- Top {top_n} Co-occurring words for '{target_word.upper()}' (within +/- {WINDOW_SIZE} window) ---")
        # Get the N most common neighbors
        top_neighbors = results[target_word].most_common(top_n)
        
        for neighbor, count in top_neighbors:
            print(f"- '{neighbor}': {count} times")
    else:
        print(f"'{target_word}' not found in the text or did not meet filtering criteria.")

In [14]:
print_top_cooccurrences(cooccurrence_results, 'සමග', top_n=5)


--- Top 5 Co-occurring words for 'සමග' (within +/- 10 window) ---
- 'හා': 77 times
- 'වූ': 60 times
- 'ඒ': 42 times
- 'නම්': 35 times
- 'විවාහ': 25 times


In [15]:
filename = 'z_score_analysis.json'

In [16]:
#5.2930

with open(filename, 'r', encoding='utf-8') as f:
        # Use json.load() to parse the JSON data into a Python object
        loaded_results: List[Dict[str, Any]] = json.load(f)

In [17]:
Z_SCORE_THRESHOLD = 5.2930

In [18]:
# Initialize an empty list to store the words that meet the criteria
highly_significant_words: List[Dict[str, Any]] = []

# Iterate through the loaded data and filter
for item in loaded_results:
    if item['z_score'] > Z_SCORE_THRESHOLD:
        highly_significant_words.append(item)

In [19]:
count_less_than_threshold = 0
for item in loaded_results:
    if item['z_score'] < Z_SCORE_THRESHOLD:
        count_less_than_threshold += 1

In [20]:
coverage_percentage = (count_less_than_threshold / len(loaded_results)) * 100
print(f"Total items: {len(loaded_results)}")
print(f"Items with -inf < Z < {Z_SCORE_THRESHOLD}: {count_less_than_threshold}")
print(f"Coverage for -inf < Z < {Z_SCORE_THRESHOLD}: {coverage_percentage:.2f}%")


Total items: 61368
Items with -inf < Z < 5.293: 61247
Coverage for -inf < Z < 5.293: 99.80%


In [21]:
highly_significant_words.sort(key=lambda x: x['z_score'], reverse=True)

In [22]:
highly_significant_words

[{'word': 'වූ', 'z_score': 140.6029636003449},
 {'word': 'ඒ', 'z_score': 70.6909537005933},
 {'word': 'ද', 'z_score': 63.19342617652523},
 {'word': 'මේ', 'z_score': 62.21972130326964},
 {'word': 'නම්', 'z_score': 60.79162082249477},
 {'word': 'යන', 'z_score': 42.291228230638495},
 {'word': 'කොට', 'z_score': 36.83848094040717},
 {'word': 'විසින්', 'z_score': 32.42435218164848},
 {'word': 'න', 'z_score': 31.807672428586606},
 {'word': 'ඇති', 'z_score': 28.919014637928345},
 {'word': 'ලද', 'z_score': 27.263716353393836},
 {'word': 'හා', 'z_score': 23.011871740177746},
 {'word': 'ය', 'z_score': 22.070623696030673},
 {'word': 'න්', 'z_score': 20.642523215255803},
 {'word': 'යයි', 'z_score': 19.60390468378317},
 {'word': 'බව', 'z_score': 19.57144785467465},
 {'word': 'ට', 'z_score': 18.889854443395734},
 {'word': 'ර', 'z_score': 18.630199810527575},
 {'word': 'වේ', 'z_score': 18.0135200574657},
 {'word': 'කියා', 'z_score': 17.299469817078265},
 {'word': 'කරණ', 'z_score': 16.68279006401639},


In [23]:
highly_significant_word_set: Set[str] = {
    item['word'] for item in highly_significant_words 
    if item['z_score'] > Z_SCORE_THRESHOLD
}

In [24]:
highly_significant_word_set # 'ධර්ම', 'ප්රකාශ', 'පිණිස', 'බුද්ධ', 'බුදු', 'බුදුන්', 'බොහෝ', 'මහ', 'මහත්', 'රජ', 'යුක්ත', 'වදාළ', 'වහන්සේ', 'වැඩ', 'ශ්රී', 'සතර', 'සඳ', 'සමග', 'සිත', 'සිත්', 'සියලු'

{'අ',
 'අප',
 'ආදි',
 'ඇති',
 'ඉතා',
 'ඊට',
 'එ',
 'එක',
 'එක්',
 'එම',
 'එය',
 'එසේ',
 'එහි',
 'ඒ',
 'ඔහු',
 'ක',
 'කර',
 'කරණ',
 'කරන',
 'කල',
 'කල්හි',
 'කළ',
 'ක්',
 'කියන',
 'කියා',
 'කී',
 'කෙරෙහි',
 'කොට',
 'ගුණ',
 'ගෙණ',
 'ගොස්',
 'ට',
 'ණ',
 'ත',
 'ත්',
 'ති',
 'තිබෙන',
 'තෙම',
 'ද',
 'දැක',
 'දැන',
 'දී',
 'ධර්ම',
 'න',
 'නම්',
 'න්',
 'නා',
 'නැත',
 'නැති',
 'නිසා',
 'නේ',
 'නො',
 'නොහොත්',
 'ප',
 'පටන්',
 'පමණ',
 'ප්රකාශ',
 'පිණිස',
 'බව',
 'බැවින්',
 'බුදු',
 'බුදුන්',
 'බොහෝ',
 'ම',
 'මට',
 'මම',
 'මහ',
 'මහත්',
 'මහා',
 'මා',
 'මෙ',
 'මෙන්',
 'මෙම',
 'මෙසේ',
 'මෙහි',
 'මේ',
 'ය',
 'යන',
 'යනු',
 'යම්',
 'යයි',
 'යි',
 'යුතු',
 'යෙ',
 'ර',
 'රජ',
 'රු',
 'ල',
 'ලද',
 'ලා',
 'ලෙස',
 'ලේ',
 'ව',
 'වදාළ',
 'වන',
 'වනාහි',
 'වශයෙන්',
 'වහන්සේ',
 'වැඩ',
 'විය',
 'විසින්',
 'වී',
 'වූ',
 'වෙයි',
 'වේ',
 'ශ්රී',
 'ස',
 'සතර',
 'සමග',
 'සහ',
 'සිටි',
 'සිත',
 'සියලු',
 'සේ',
 'සේක',
 'හට',
 'හා',
 'හැර',
 'හෙයින්',
 'හෙවත්',
 'හෝ'}

In [25]:
identified_non_stopwords = ['ධර්ම', 'ප්රකාශ', 'පිණිස', 'බුද්ධ', 'බුදු', 'බුදුන්', 'බොහෝ', 'මහ', 'මහත්', 'රජ', 'යුක්ත', 'වදාළ', 'වහන්සේ', 'වැඩ', 'ශ්රී', 'සතර', 'සඳ', 'සමග', 'සිත', 'සිත්', 'සියලු']

In [26]:
highly_significant_word_set = [item for item in highly_significant_word_set if item not in identified_non_stopwords]

In [27]:
highly_significant_word_set

['වන',
 'යන',
 'වනාහි',
 'එම',
 'සහ',
 'මා',
 'නො',
 'ස',
 'නැති',
 'මහා',
 'පමණ',
 'මෙහි',
 'කියන',
 'විසින්',
 'එක්',
 'තෙම',
 'පටන්',
 'යෙ',
 'නැත',
 'ප',
 'කොට',
 'කරණ',
 'ඔහු',
 'හට',
 'ති',
 'වේ',
 'නේ',
 'කරන',
 'ය',
 'දැක',
 'ඇති',
 'ගුණ',
 'ලද',
 'කල',
 'කී',
 'සේක',
 'යනු',
 'ක්',
 'ර',
 'සේ',
 'හෝ',
 'රු',
 'එහි',
 'ට',
 'එ',
 'අප',
 'මේ',
 'වෙයි',
 'නම්',
 'ද',
 'ම',
 'හැර',
 'යුතු',
 'එය',
 'නිසා',
 'බැවින්',
 'තිබෙන',
 'මෙසේ',
 'න්',
 'වූ',
 'විය',
 'ගොස්',
 'මට',
 'ව',
 'වී',
 'මෙන්',
 'ගෙණ',
 'මෙ',
 'අ',
 'සිටි',
 'ඊට',
 'යි',
 'වශයෙන්',
 'ණ',
 'ත',
 'හා',
 'නොහොත්',
 'ත්',
 'නා',
 'කර',
 'කෙරෙහි',
 'ඉතා',
 'එක',
 'කියා',
 'බව',
 'මෙම',
 'කල්හි',
 'ක',
 'ඒ',
 'මම',
 'කළ',
 'හෙයින්',
 'එසේ',
 'ල',
 'හෙවත්',
 'ලේ',
 'යම්',
 'ලා',
 'දැන',
 'දී',
 'ආදි',
 'ලෙස',
 'යයි',
 'න']

In [28]:
filtered_cooccurrence_map: Dict[str, Counter] = {}

for target_word, neighbors_counter in cooccurrence_results.items():
    
    # A. Target word is NOT SKIPPED, regardless of its significance.
    #    The original line 'if target_word in highly_significant_word_set: continue' is removed.
    
    # B. Filter its NEIGHBORS (Values)
    filtered_neighbors_counter = Counter()
    
    for neighbor, count in neighbors_counter.items():
        # Check if the NEIGHBOR word is highly significant
        if neighbor not in highly_significant_word_set:
            # If the neighbor is NOT significant, keep it
            filtered_neighbors_counter[neighbor] = count
            
    # C. Only save the target word if it still has non-empty neighbors after filtering
    if filtered_neighbors_counter:
        filtered_cooccurrence_map[target_word] = filtered_neighbors_counter


In [29]:
filtered_cooccurrence_map['දරුවන්']

Counter({'රජ': 25,
         'වැදැහැ': 8,
         'ඔවුන්': 7,
         'පුත්ර': 7,
         'ස්වාමි': 5,
         'සමග': 5,
         'විවාහ': 5,
         'ගෑනු': 5,
         'මින්': 5,
         'විනාස': 5,
         'සෙසු': 4,
         'ඇගේ': 4,
         'දොන්': 4,
         'මහතා': 4,
         'පිරිමි': 4,
         'දෙදෙනකු': 4,
         'ලැබ': 4,
         'අදර්ශනයට': 4,
         'අත්': 3,
         'පා': 3,
         'ගත': 3,
         'අඹු': 3,
         'ලබා': 3,
         'තමන්': 3,
         'පිණිස': 3,
         'තොමෝ': 3,
         'කුරුප්පු': 3,
         'ජන්මාන්තර': 3,
         'වැඩිමහලු': 3,
         'ලැබුවාය': 3,
         'මේරි': 3,
         'ලත්': 3,
         'මූලොච්ඡින්න': 3,
         'කෙරෙමි': 3,
         'හැ': 3,
         'පමුනුවමි': 3,
         'මහාපරිනිර්වාණ': 3,
         'සූත්රය': 3,
         'අවැඩට': 3,
         'යුත්තේය': 3,
         'උගන්නා': 3,
         'ව්යාකරණ': 3,
         'පිය': 2,
         'යෙහි': 2,
         'බාල': 2,
         'හිස්': 2,
         'දෙදෙන': 2,
        

## Century-wise Analaysis

In [30]:
list_of_file_locations['written_date'].replace(['', None], np.nan, inplace=True)

/var/folders/p0/y_n4880x7cjbhx11tzvvqpzh0000gn/T/ipykernel_85333/1470307445.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  list_of_file_locations['written_date'].replace(['', None], np.nan, inplace=True)


In [31]:
list_of_file_locations.dropna(subset=['written_date'], inplace=True)

In [32]:
list_of_file_locations.head()

,written_date,text_file_location
0,1187 - 1225,/Users/nevidujayatilleke/Documents/MSC - Resea...
2,1832 - 1907,/Users/nevidujayatilleke/Documents/MSC - Resea...
3,1901 - 1938,/Users/nevidujayatilleke/Documents/MSC - Resea...
4,1944,/Users/nevidujayatilleke/Documents/MSC - Resea...
5,1303 - 1333,/Users/nevidujayatilleke/Documents/MSC - Resea...


In [33]:
len(list_of_file_locations)

59

In [34]:
list_of_file_locations.reset_index(drop=True, inplace=True)

In [35]:
list_of_file_locations.head()

,written_date,text_file_location
0,1187 - 1225,/Users/nevidujayatilleke/Documents/MSC - Resea...
1,1832 - 1907,/Users/nevidujayatilleke/Documents/MSC - Resea...
2,1901 - 1938,/Users/nevidujayatilleke/Documents/MSC - Resea...
3,1944,/Users/nevidujayatilleke/Documents/MSC - Resea...
4,1303 - 1333,/Users/nevidujayatilleke/Documents/MSC - Resea...


In [36]:
list_of_file_locations['start_date'] = np.nan
list_of_file_locations['end_date'] = np.nan

In [37]:
for index, row in list_of_file_locations.iterrows():
    date_str = str(row['written_date']) # Ensure it's treated as a string
    
    start_year = np.nan
    end_year = np.nan

    if ' - ' in date_str:
        # It's a range
        try:
            start_year_str, end_year_str = date_str.split(' - ')
            start_year = int(start_year_str.strip())
            end_year = int(end_year_str.strip())
        except ValueError:
            print(f"Warning: Could not parse range '{date_str}' at index {index}. Setting to NaN.")
    else:
        # It's a single year or potentially unparseable
        try:
            single_year = int(date_str.strip())
            start_year = single_year
            end_year = single_year # Duplicate for single date
        except ValueError:
            print(f"Warning: Could not parse single year/date '{date_str}' at index {index}. Setting to NaN.")
            
    # Assign the parsed (or NaN) values back to the DataFrame
    list_of_file_locations.at[index, 'start_date'] = start_year
    list_of_file_locations.at[index, 'end_date'] = end_year


In [38]:
list_of_file_locations.head()

,written_date,text_file_location,start_date,end_date
0,1187 - 1225,/Users/nevidujayatilleke/Documents/MSC - Resea...,1187.0,1225.0
1,1832 - 1907,/Users/nevidujayatilleke/Documents/MSC - Resea...,1832.0,1907.0
2,1901 - 1938,/Users/nevidujayatilleke/Documents/MSC - Resea...,1901.0,1938.0
3,1944,/Users/nevidujayatilleke/Documents/MSC - Resea...,1944.0,1944.0
4,1303 - 1333,/Users/nevidujayatilleke/Documents/MSC - Resea...,1303.0,1333.0


In [39]:
def get_century(year):
    """Calculates the century for a given year."""
    if pd.isna(year):
        return np.nan
    return (int(year) - 1) // 100 + 1

In [40]:
list_of_file_locations['end_century'] = list_of_file_locations['end_date'].apply(get_century)

In [41]:
list_of_file_locations.head()

,written_date,text_file_location,start_date,end_date,end_century
0,1187 - 1225,/Users/nevidujayatilleke/Documents/MSC - Resea...,1187.0,1225.0,13
1,1832 - 1907,/Users/nevidujayatilleke/Documents/MSC - Resea...,1832.0,1907.0,20
2,1901 - 1938,/Users/nevidujayatilleke/Documents/MSC - Resea...,1901.0,1938.0,20
3,1944,/Users/nevidujayatilleke/Documents/MSC - Resea...,1944.0,1944.0,20
4,1303 - 1333,/Users/nevidujayatilleke/Documents/MSC - Resea...,1303.0,1333.0,14


In [42]:
list_of_file_locations['tokens'] = list_of_file_locations['text_file_location'].apply(clean_words)

In [43]:
list_of_file_locations['tokens'][0][:10]

['අප',
 'බුදුන්',
 'සාරාසකි',
 'කප්සුවහස්',
 'මතුයෙහි',
 'කුලුණුනු',
 'වණින්',
 'යුත්',
 'මහත',
 'වු']

In [44]:
DOCUMENT_SEPARATOR = "__DOC_END__" 

In [45]:
def combine_tokens_with_separator(series_of_lists):
    """
    Takes a pandas Series of token lists and combines them, 
    inserting a separator between each list.
    """
    combined_list = []
    
    # Iterate over each document's token list (item) in the Series
    for i, token_list in enumerate(series_of_lists):
        # 1. Add the tokens from the current document
        combined_list.extend(token_list)
        
        # 2. Add the separator, but only if it's not the last document in the group
        if i < len(series_of_lists) - 1:
            combined_list.append(DOCUMENT_SEPARATOR)
            
    return combined_list

In [46]:
century_tokens = list_of_file_locations.groupby('end_century')['tokens'].apply(combine_tokens_with_separator)
century_tokens = century_tokens.to_dict()

In [47]:
century_tokens

{5: ['කාම',
  'ලොක',
  'රූප',
  'ලොක',
  'අරූපලෝක',
  'යන්',
  'ලොකත්රයට',
  'ශ්රෙෂ්ඨ',
  'වූ',
  'මුනීන්ද්ර',
  'නම්',
  'වූ',
  'සම්යක්',
  'සම්බුද්ධයන්',
  'වහන්සේගේ',
  'ශ්රී',
  'පාද',
  'පද්මය',
  'භය',
  'ලොහ',
  'කුලාචාරයෙන්',
  'විනා',
  'සකසා',
  'වැඳ',
  'සකල',
  'ශාස්ත්රයෙහි',
  'පාරප්රාප්ත',
  'පණ්ඩිත',
  'වූ',
  'භෛෂජ්යාර්ථයෙහි',
  'දක්ෂ',
  'වූ',
  'වෙදුන්',
  'විසින්',
  'උතුම්',
  'වූ',
  'රස',
  'සංහිතාදි',
  'තන්ත්ර',
  'ශාස්ත්රයෙහි',
  'වුත',
  'කියන',
  'ලද',
  'විවිධං',
  'නානාප්රකාර',
  'වූ',
  'සාරාර්ථය',
  'උදුරා',
  'ගෙණ',
  'සකල',
  'සත්වයන්ගේ',
  'රොග',
  'ව්යපසමන',
  'සංඛ්යාත',
  'සංග්රහව',
  'පිණිස',
  'මේ',
  'සාරාර්ථ',
  'සංග්රහ',
  'නම්',
  'ප්රකරණය',
  'සර්ථාවබොධය',
  'සඳහා',
  'කියන්නෙමි',
  'යනාදී',
  'අනෙකවිධ',
  'මානයෙන්',
  'දැක්වූ',
  'මේ',
  'වෛද්ය',
  'ශාස්ත්රය',
  'ප්රථම',
  'කල්පයෙහි',
  'ලෝකාර්ථය',
  'කරන්නා',
  'වූ',
  'මහ',
  'බ්රහ්මයා',
  'දක්ෂ',
  'පුජාපති',
  'හට',
  'කීහ',
  'දක්ෂ',
  'ප්රජාපති',
  'අග',
  'විනීන්',
  'දෙදෙනා',
  'හට',

In [48]:
century_tokens[12]

['තමා',
 'වරදස',
 'නොදිස්',
 'නේ',
 'මෙරමා',
 'දොස්',
 'දිස්නේ',
 'නුවන්',
 'බැහැර',
 'නහමත්',
 'තමා',
 'මුත්',
 'නොදක්නේ',
 'කිම්',
 'සරසවිය',
 'බැලුමසෙක්',
 'නි',
 'වෙත්වා',
 'කිවි',
 'දෙනෝ',
 'පෙදෙහි',
 'රසහව්',
 'විදුනා',
 'දෙනෙතා',
 'ඉතා',
 'දුලබෝ',
 'ම',
 'තෙපල',
 'පා',
 'ලදපිළි',
 'වේ',
 'වෙසෙස්',
 'මෙ',
 'සෙබේ',
 'කසුන්',
 'කෙවුන්නේ',
 'බඳ',
 'මිණීමැ',
 'ගුණෙනුතුරත්',
 'වේ',
 'තමා',
 'වරදස',
 'නොදිස්නේ',
 'තමාගේ',
 'වරදෙහි',
 'ස්වල්පයකුදු',
 'අදෘශ්යය',
 'මෙරමා',
 'දොස්',
 'දිස්නේ',
 'අනුන්ගේ',
 'දොෂ',
 'වනාහි',
 'පැණෙන්නේන්',
 'මැයි',
 'නුවන්',
 'නෙත්රය',
 'තමා',
 'මුත්',
 'තමා',
 'විනා',
 'බැහැර',
 'නහමත්',
 'බාහ්ය',
 'වස්තූන්',
 'විෂයෙහි',
 'විචාරත්',
 'හොත්',
 'නොදක්නේ',
 'කිම්',
 'සර්ෂප',
 'ආදී',
 'කොට',
 'අදෘශ්යමාන',
 'කිමෙක්ද',
 'සරසවිය',
 'සරස්වතියගේ',
 'බැලුමසෙක්නි',
 'අර්ධ',
 'නිරීක්ෂණ',
 'මාත්රයෙකින්',
 'දෙනෝ',
 'කිවි',
 'වෙත්වා',
 'ජනයෝ',
 'කිවිහු',
 'වෙත්වා',
 'පෙදෙහි',
 'රසහව්',
 'විදුනා',
 'දෙනෙතා',
 'පදයෙහි',
 'රසභාවය',
 'දන්නා',
 'ජනයෝ',
 'වනාහි',
 'ඉතා',
 'දුලබ

In [49]:
from typing import List, Dict, Counter as T_Counter, Set, Any

In [50]:
def find_century_wise_context_cooccurrences(tokens: List[str], window: int, separator: str) -> Dict[str, T_Counter]:
    """
    Finds all words co-occurring within a +/- window of a target word, 
    respecting document boundaries defined by a separator token.
    """
    cooccurrence_map = {}
    
    # Iterate through every token in the century's text
    for i, target_word in enumerate(tokens):
        
        # 1. Skip the separator token itself (it should never be a target word)
        if target_word == separator:
            continue
            
        # Initialize a counter for this target word if it doesn't exist
        if target_word not in cooccurrence_map:
            cooccurrence_map[target_word] = Counter()
            
        # Define the search range
        start_index = max(0, i - window)
        end_index = min(len(tokens), i + window + 1)
        
        # Extract the context window slice
        context_window = tokens[start_index:end_index]
        
        # Add all valid neighbors in the window
        for neighbor in context_window:
            # 2. Skip the target word itself
            if neighbor == target_word:
                continue
            
            # 3. CRUCIAL: Skip the separator token when counting neighbors
            if neighbor == separator:
                continue

            if neighbor in highly_significant_word_set:
                continue
                
            cooccurrence_map[target_word][neighbor] += 1
                
    return cooccurrence_map

In [51]:
# Dictionary to store the co-occurrence results for every century
century_cooccurrence_results: Dict[str, Dict[str, T_Counter]] = {}

print(f"Starting co-occurrence analysis (Window: +/- {WINDOW_SIZE}) for each century...")

for century, tokens in century_tokens.items():
    print(f"Analyzing {century} century (Total tokens: {len(tokens)})...")
    
    # Run the analysis for the current century's tokens
    cooccurrence_map = find_century_wise_context_cooccurrences(
        tokens=tokens,
        window=WINDOW_SIZE,
        separator=DOCUMENT_SEPARATOR
    )
    
    # Save the result
    century_cooccurrence_results[century] = cooccurrence_map

Starting co-occurrence analysis (Window: +/- 10) for each century...
Analyzing 5 century (Total tokens: 565)...
Analyzing 12 century (Total tokens: 866)...
Analyzing 13 century (Total tokens: 13626)...
Analyzing 14 century (Total tokens: 5213)...
Analyzing 15 century (Total tokens: 8490)...


Analyzing 16 century (Total tokens: 1369)...
Analyzing 18 century (Total tokens: 4710)...
Analyzing 19 century (Total tokens: 12707)...
Analyzing 20 century (Total tokens: 17309)...


In [52]:
century_cooccurrence_results[13]['දෙමවුපියන්']

Counter({'තමන්': 2,
         'සිත්': 1,
         'ඇතිව': 1,
         'අනුන්ට': 1,
         'අනුශාසනා': 1,
         'කලත්': 1,
         'යහපත්': 1,
         'තරම්': 1,
         'දරුවන්': 1,
         'තරමේ': 1,
         'හික්ම': 1,
         'වන්නා': 1,
         'සියලු': 1,
         'අකුසලින්': 1,
         'දුරුව': 1})

In [53]:
len(century_cooccurrence_results[13])

5883

In [54]:
# all_centuries = list(century_cooccurrence_results.keys())

# for century in all_centuries[9:]:
#     current_century_keys = set(century_cooccurrence_results[century].keys())
#     # Find the intersection (words present in both the current common_keys set and the current century)
#     common_keys = common_keys.intersection(current_century_keys)


In [55]:
def get_consistent_keys(
    full_cooccurrence_results: Dict[str, Dict[str, T_Counter]], 
    selected_centuries: List[str]
) -> Set[str]:
    """Finds the intersection of target words across selected centuries."""

    if not selected_centuries:
        return set()

    all_key_sets = []
    
    for century in selected_centuries:
        if century not in full_cooccurrence_results:
            print(f"Error: Century '{century}' not found. Returning empty set.")
            return set()
        all_key_sets.append(set(full_cooccurrence_results[century].keys()))
    
    if not all_key_sets:
        return set()
        
    return set.intersection(*all_key_sets)

In [56]:
centuries_to_compare = [13, 14, 15 ,16, 18, 19, 20]
# Find the keys consistent ONLY within the selected subset
consistent_keys = get_consistent_keys(century_cooccurrence_results, centuries_to_compare)

In [57]:
consistent_keys

{'අප',
 'අසා',
 'ආ',
 'ආදි',
 'ඇති',
 'ඊට',
 'එ',
 'එක',
 'එක්',
 'එම',
 'එසේ',
 'ඒ',
 'ඔහු',
 'කර',
 'කල',
 'කළ',
 'කාල',
 'කියන',
 'කිසි',
 'කී',
 'කොට',
 'ගත',
 'ගිය',
 'ගුණ',
 'ද',
 'දක්වා',
 'දත්',
 'දවස',
 'දැක',
 'දී',
 'දුක්',
 'දෙන',
 'දෙවියන්',
 'නම්',
 'නැති',
 'නිසි',
 'නොයෙක්',
 'නොව',
 'පද',
 'පමණ',
 'පළමු',
 'පැමිණි',
 'පුර',
 'පෙර',
 'බල',
 'බලා',
 'මතු',
 'මල්',
 'මහ',
 'මහත්',
 'මා',
 'මෙන්',
 'මෙසේ',
 'මේ',
 'යන',
 'යනු',
 'යම්',
 'යුතු',
 'රජ',
 'රැස්',
 'ලද',
 'වන',
 'වසන',
 'විය',
 'විසිනුදු',
 'වී',
 'වූ',
 'වෙන',
 'සතර',
 'සත්',
 'සත්ව',
 'සමග',
 'සිට',
 'සිටි',
 'සිත',
 'සියලු',
 'හට',
 'හා',
 'හැර',
 'හෝ'}

In [58]:
len(consistent_keys)

80

In [59]:
filtered_century_cooccurrence_results: Dict[str, Dict[str, T_Counter]] = {}

In [60]:
filtered_cooccurrence_map_keys: Dict[str, T_Counter] = {
    target_word: neighbors_counter
    for target_word, neighbors_counter in cooccurrence_results.items()
    if target_word not in highly_significant_word_set
}

In [61]:
for century in centuries_to_compare:
    cooccurrence_map = century_cooccurrence_results.get(century)
    
    if cooccurrence_map is None:
        continue 

    # Filter the map to keep only the consistent keys
    filtered_map = {
        key: cooccurrence_map[key] 
        for key in consistent_keys
    }
    
    filtered_century_cooccurrence_results[century] = filtered_map

In [62]:
len(filtered_century_cooccurrence_results[14])

80

In [63]:
for century, filtered_map in filtered_century_cooccurrence_results.items():
        print(f"[{century}]:")
        for word, neighbors in filtered_map.items():
            print(f"  Target: '{word}' -> Neighbors (Top 3): {neighbors.most_common(3)}")

[13]:
  Target: 'වන' -> Neighbors (Top 3): [('විෂයාතික්රාන්තයහ', 8), ('බුදු', 5), ('වාග්', 4)]
  Target: 'යන' -> Neighbors (Top 3): [('ධාතුයැ', 38), ('අර්ථයි', 24), ('කෘදන්තයෙන්', 22)]
  Target: 'එම' -> Neighbors (Top 3): [('අධිගතයන්', 4), ('දශක', 3), ('සමුත්ථාන', 3)]
  Target: 'මා' -> Neighbors (Top 3): [('හිමියන්', 5), ('වැ', 4), ('නැමැති', 4)]
  Target: 'නැති' -> Neighbors (Top 3): [('කෙනෙකුන්', 4), ('තමා', 3), ('රූප', 3)]
  Target: 'පමණ' -> Neighbors (Top 3): [('ලක්ෂයක්', 7), ('කෙළ', 5), ('යනුයෙන්', 3)]
  Target: 'කියන' -> Neighbors (Top 3): [('සමාධි', 4), ('ලදී', 3), ('යනාදි', 3)]
  Target: 'එක්' -> Neighbors (Top 3): [('ගුරු', 4), ('යස', 4), ('දවසක්', 3)]
  Target: 'කොට' -> Neighbors (Top 3): [('වැ', 10), ('සියලු', 9), ('ඇත්තේයි', 9)]
  Target: 'ඔහු' -> Neighbors (Top 3): [('සකල', 2), ('දැනැ', 2), ('නිමිත්තෙන්', 2)]
  Target: 'හට' -> Neighbors (Top 3): [('සඳහා', 5), ('වැ', 4), ('දුක', 4)]
  Target: 'සිට' -> Neighbors (Top 3): [('මාරවිජය', 1), ('සර්වඥපදප්රාප්ත', 1), ('වැ', 1)]
  T

In [64]:
final_filtered_results: Dict[str, Dict[str, T_Counter]] = {}

for century, cooccurrence_map in filtered_century_cooccurrence_results.items():
    
    # Filter the map to remove any target words found in the highly significant set
    doubly_filtered_map = {
        target_word: neighbors_counter
        for target_word, neighbors_counter in cooccurrence_map.items()
        if target_word not in highly_significant_word_set
    }
    
    # Only store the result if the map is not empty after filtering
    if doubly_filtered_map:
        final_filtered_results[century] = doubly_filtered_map

In [65]:
len(final_filtered_results[13].keys())

38

In [66]:
final_filtered_results[13]['සතර']

Counter({'ගණ': 5,
         'පර්වතය': 3,
         'භාගයෙහි': 3,
         'මනා': 3,
         'ඉෂ්ට': 3,
         'මෙසේයි': 3,
         'ඍතු': 3,
         'මණ්ඩලය': 2,
         'වැනි': 2,
         'භාගයෙන්': 2,
         'දෙභාගයෙක්': 2,
         'අක්': 2,
         'පතුල්': 2,
         'තුන්වන': 2,
         'ජංඝා': 2,
         'ප්රතිෂඨිත': 2,
         'ත්රිවිධ': 2,
         'යනාදි': 2,
         'සමාධි': 2,
         'උසස්': 2,
         'ස්ථානයකට': 2,
         'පැමිණේ': 2,
         'කිමවන': 2,
         'ලදී': 2,
         'අනිෂ්ට': 2,
         'දෙඅත්හි': 2,
         'ලුහු': 2,
         'දෙකින්': 2,
         'යන්නෙහි': 2,
         'පාරාජිකාවෝ': 2,
         'ච': 2,
         'හෘදය': 2,
         'ඉක්බිත්තෙන්': 2,
         'වා': 2,
         'ධාතූන්': 2,
         'සමුත්ථාන': 2,
         'රූපයො': 2,
         'වසයෙන්': 2,
         'සොළොස්': 2,
         'පැසුණු': 2,
         'හිමවත්': 1,
         'මෙරු': 1,
         'සුර්යය': 1,
         'චන්ද්ර': 1,
         'සපරිවාර': 1,
         'මහාද්වීපය': 1,
    

In [67]:
final_filtered_results[16]['මල්']

Counter({'යක්': 2,
         'මනෝ': 1,
         'රෝගෙට': 1,
         'වේය': 1,
         'අවුසද': 1,
         'අකුරෙව්': 1,
         'සතර': 1,
         'දක්වා': 1,
         'මෙයින්': 1,
         'අනතුර': 1,
         'පලුගත්': 1,
         'පතර': 1,
         'තුරක්ෂ': 1,
         'දක්වනෙම්': 1,
         'ගෙනහැර': 1,
         'දුත': 1,
         'ලකුණුයන': 1,
         'පෙර': 1,
         'මග': 1,
         'ඇදුරාටද': 1,
         'පසු': 1,
         'දෙනක්හෝ': 1,
         'දි': 1,
         'කක්': 1,
         'සුදු': 1,
         'දමක්හෝ': 1,
         'ගජසිඳු': 1,
         'ගොනෙක්': 1,
         'පිරිකුඹු': 1,
         'බෙරක්හෝ': 1,
         'නිළ': 1,
         'නිසි': 1})

In [68]:
final_filtered_results[13]['මල්']

Counter({'නැමැති': 4,
         'අතුපතර': 1,
         'ත්රිවිධ': 1,
         'චර්යයා': 1,
         'පල්ලව': 1,
         'අභිඥා': 1,
         'බොධිසම්භාර': 1,
         'කප්රුක': 1,
         'සර්වඥතා': 1,
         'ඥාන': 1,
         'අමාඵල': 1,
         'ගන්වයි': 1,
         'බුදුවෙයි': 1})

In [69]:
final_filtered_results[18]['මල්']

Counter({'බඳු': 5,
         'වුවමනා': 4,
         'නිල්': 3,
         'තන්හි': 3,
         'පැහැය': 3,
         'රන්වන්': 3,
         'බුදුන්ගේ': 2,
         'ලක්ෂණයෙන්': 2,
         'දිය': 2,
         'බෙරලිය': 2,
         'කිණිහිරි': 2,
         'ජංඝාව': 1,
         'හාත්පසින්': 1,
         'පිරි': 1,
         'ගිය': 1,
         'හැගැබිසෙයින්': 1,
         'සම්': 1,
         'පූර්ණය': 1,
         'වටවූ': 1,
         'තැඹිලි': 1,
         'කැකුළු': 1,
         'සෙයින්': 1,
         'සම්පූර්ණ': 1,
         'ජංඝාවන්': 1,
         'එනිජංඝා': 1,
         'ස්වර': 1,
         'කරවීක': 1,
         'භාණි': 1,
         'සුදු': 1,
         'තන්හී': 1,
         'දහවත්තරු': 1})

In [70]:
final_filtered_results[19]['සතර']

Counter({'පාද': 8,
         'රූප': 4,
         'තැනින්': 3,
         'සතරවන': 3,
         'ස්ථානයකින්': 3,
         'උගුළුවාගෙන': 3,
         'සම': 3,
         'යුගලයෙහි': 3,
         'කව්': 3,
         'අයිති': 3,
         'ස්කන්ධ': 3,
         'නාම': 3,
         'දෙකට': 3,
         'විෂමපාදය': 2,
         'දෙයානුවන': 2,
         'සමපාදය': 2,
         'වර්ණයකින්': 2,
         'පරව': 2,
         'ජගණයක්': 2,
         'පාදය': 2,
         'දෙක': 2,
         'අනුවන': 2,
         'යනාදි': 2,
         'පාඨයෙන්': 2,
         'කීවොත්': 2,
         'විඤ්ඤාණයට': 2,
         'තිස්වන': 1,
         'තුන්වන': 1,
         'කොටගෙන': 1,
         'පථ්යාව': 1,
         'කත්ර': 1,
         'අසම': 1,
         'ආද්යන්තලඝු': 1,
         'ගුරුද්වය': 1,
         'භගණජගණද්වයකින්': 1,
         'සම්බන්ධ': 1,
         'වු': 1,
         'ගණ': 1,
         'අටකින්ද': 1,
         'අක්ෂර': 1,
         'දෙතිසකින්ද': 1,
         'මාත්රා': 1,
         'එකුන්': 1,
         'පණසකින්ද': 1,
         'ගුරු': 1,
         'වියර

In [71]:
records = []
for century in centuries_to_compare:
    for target_word, neighbors_counter in final_filtered_results[century].items():
        for neighbor_word, count in neighbors_counter.items():
            records.append({
                'target_word': target_word,
                'collocate_word': neighbor_word,
                'cooccurrence_count': count
            })
    temp_df = pd.DataFrame(records)
    temp_df.to_csv(f'Collocates/collocates_{century}th_century.csv', index=False, encoding='utf-8-sig')       

In [72]:
TARGET_WORD = 'සතර'
records: List[Dict[str, Any]] = []

In [73]:
common_keys = list(final_filtered_results[13].keys())

In [74]:
final_filtered_results[13].items()

dict_items([('සිට', Counter({'මාරවිජය': 1, 'සර්වඥපදප්රාප්ත': 1, 'වැ': 1, 'හුනස්නෙන්': 1, 'නැඟී': 1, 'පූර්වොත්තර': 1, 'දිශා': 1, 'භාගයෙහි': 1, 'වැඩ': 1, 'සප්තාහයක්': 1, 'අනිමිසලෝචන': 1, 'යුගලයෙන්': 1, 'පුදන': 1, 'ජය': 1, 'බොධිය': 1, 'තත': 1, 'ප්රභව': 1, 'දෙසන': 1, 'ලදි': 1, 'ගාථායෙහි': 1, 'සංක්ෂෙප': 1, 'වර්ණනා': 1, 'යන්නෙහි': 1, 'ශීලයෙහි': 1, 'ශීලය': 1, 'පුරන්නෙ': 1, 'සර්ව': 1, 'භාගයෙන්': 1, 'සංවරණය': 1, 'කරන්': 1, 'ව්යතික්රම': 1, 'නොකරන්නේය': 1, 'ආරාමාදි': 1, 'වස්තුව': 1, 'සොර': 1, 'සිතින්': 1, 'සටන්': 1, 'බැණ': 1, 'ගණී': 1, 'යමින්': 1, 'ගනී': 1, 'තමා': 1, 'නගා': 1, 'තුබුවා': 1, 'මම්': 1, 'නොගන්මි': 1, 'යනා': 1, 'දීන්': 1})), ('මතු', Counter({'වැසී': 4, 'පැවැති': 4, 'ගේ': 3, 'වතුරින්': 3, 'වන්නට': 3, 'නැමැති': 2, 'මෙයින්': 2, 'නොවෙයි': 2, 'සමාධි': 2, 'ධර්ම': 2, 'යනාදි': 2, 'පදාර්ථ': 2, 'ආයේ': 2, 'අවදියෙකි': 2, 'කලක්': 2, 'පැවැතී': 2, 'ගත්': 2, 'පියෙසෙක්හි': 2, 'සරියේ': 2, 'මිනිසා': 2, 'වැඩට': 2, 'සුජාත': 1, 'මවු': 1, 'පිය': 1, 'පක්ෂ': 1, 'ජාතිවාදයෙන්': 1, 'අකලිෂ්ට': 1, 'ඇතිවන': 1, 'දෙත

In [75]:
# for key in common_keys:   
#     for century, cooccurrence_map in final_filtered_results.items():
#         if TARGET_WORD in cooccurrence_map:
#             neighbors_counter = cooccurrence_map[key]
#             # print(neighbors_counter)
            
#             for collocate_word, count in neighbors_counter.items():
#                 records.append({
#                     'century': century,
#                     'collocate_word': collocate_word,
#                     # Temporary column to mark presence
#                     'count': count
#                 })
#     temp_df = pd.DataFrame(records)
#     # print(temp_df)
#     temp_df = temp_df.pivot_table(index='collocate_word', columns='century', values='count', aggfunc='first', fill_value=' ')
#     temp_df.to_csv(f'Collocates/{key}.csv', index=True, encoding='utf-8-sig')

In [76]:
for key in common_keys:   
    records = []  # ← RESET HERE, inside the loop
    for century, cooccurrence_map in final_filtered_results.items():
        if key in cooccurrence_map:          # ← also fix: check for `key`, not TARGET_WORD
            neighbors_counter = cooccurrence_map[key]
            
            for collocate_word, count in neighbors_counter.items():
                records.append({
                    'century': century,
                    'collocate_word': collocate_word,
                    'count': count
                })
    temp_df = pd.DataFrame(records)
    temp_df = temp_df.pivot_table(
        index='collocate_word',
        columns='century',
        values='count',
        aggfunc='sum',   # ← use 'sum' instead of 'first'
        fill_value=0     # ← use 0 instead of ' ' for cleaner numeric output
    )
    temp_df.to_csv(f'Collocates/{key}.csv', index=True, encoding='utf-8-sig')

## 'මහ' Analysis

In [77]:
maha_collocates_df = pd.read_csv('Collocates/මහ.csv')

In [78]:
maha_collocates_df.head()

,collocate_word,13,14,15,16,18,19,20
0,අක්ෂර,0,0,0,0,0,0,1
1,අකැප,2,0,0,0,0,0,0
2,අචින්ත්ය,0,1,0,0,0,0,0
3,අජාසත්,0,2,0,0,0,0,0
4,අඤ්ඤාකොණ්ඩඤ්ඤ,0,0,0,0,0,1,0


In [79]:
maha_collocates_df = maha_collocates_df.replace(r'^\s*$', np.nan, regex=True).fillna(0)

In [80]:
maha_collocates_df.isna().sum()

collocate_word    0
13                0
14                0
15                0
16                0
18                0
19                0
20                0
dtype: int64

In [81]:
cols_to_sum = ['13', '14', '15', '16', '18', '19', '20']
maha_collocates_df[cols_to_sum] = maha_collocates_df[cols_to_sum].apply(pd.to_numeric, errors='coerce')
maha_collocates_df['total_count'] = maha_collocates_df[cols_to_sum].sum(axis=1)

In [82]:
maha_collocates_df.sort_values(by='total_count', ascending=False, inplace=True)

In [83]:
maha_collocates_df.head()

,collocate_word,13,14,15,16,18,19,20,total_count
442,නැමැති,10,0,1,0,0,0,0,11
838,වහන්සේ,1,6,0,0,0,2,2,11
463,නිවන්,8,0,2,0,0,0,0,10
741,රජ,0,1,0,0,0,0,8,9
64,අසා,3,2,0,1,1,0,0,7


In [84]:
maha_collocates_df['century_span'] = (maha_collocates_df[cols_to_sum] > 0).sum(axis=1)

In [85]:
maha_collocates_df['longevity_score'] = maha_collocates_df['total_count'] * maha_collocates_df['century_span']

In [86]:
maha_collocates_df.sort_values(by='longevity_score', ascending=False, inplace=True)

In [87]:
maha_collocates_df.head()

,collocate_word,13,14,15,16,18,19,20,total_count,century_span,longevity_score
838,වහන්සේ,1,6,0,0,0,2,2,11,4,44
64,අසා,3,2,0,1,1,0,0,7,4,28
442,නැමැති,10,0,1,0,0,0,0,11,2,22
753,රහතන්,0,2,0,0,0,1,4,7,3,21
463,නිවන්,8,0,2,0,0,0,0,10,2,20


In [88]:
maha_collocates_df.to_csv('Collocates/maha_collocates_analysis.csv', index=False, encoding='utf-8-sig')

In [89]:
maha_collocates_df[maha_collocates_df['collocate_word'] == 'ප්රධාන']

,collocate_word,13,14,15,16,18,19,20,total_count,century_span,longevity_score
526,ප්රධාන,1,0,0,0,0,0,1,2,2,4


In [90]:
# maha_collocates_df['collocate_word']

In [91]:
maha_collocates = ['වහන්සේ', 'රහතන්', 'තෙරුන්', 'සඟ', 'වහන්සේගේ', 'ධර්ම', 'සංඝයා', 'බුදුන්', #esteemed, great, sacred
                   'රජ', 'ප්රධාන', 'රජුන්', 'ඉසුරු', 'ඇමැති', 'රජානනි', 'රාජගුරු', 'අජාසත්' ,'නරනිඳු', 'ධර්මාශෝක', 'පතිරජ', #powerful, royal, political
                   'ලක්ෂයක්', 'මහත්', 'සමූහයක්', 'මුහුද', 'කෛලාස' #big, large, massive
                   ] 

In [92]:
maha_collocates_df[maha_collocates_df['collocate_word'].isin(maha_collocates)]

,collocate_word,13,14,15,16,18,19,20,total_count,century_span,longevity_score
838,වහන්සේ,1,6,0,0,0,2,2,11,4,44
753,රහතන්,0,2,0,0,0,1,4,7,3,21
741,රජ,0,1,0,0,0,0,8,9,2,18
775,ලක්ෂයක්,4,1,0,0,0,0,1,6,3,18
937,සඟ,0,2,1,0,0,0,2,5,3,15
337,තෙරුන්,1,3,0,0,0,1,0,5,3,15
412,ධර්ම,1,1,0,0,0,0,1,3,3,9
839,වහන්සේගේ,1,3,0,0,0,0,0,4,2,8
931,සංඝයා,0,0,2,0,1,0,0,3,2,6
627,බුදුන්,2,0,0,0,0,0,1,3,2,6


In [93]:
meanings_map = {
    **{w: "esteemed, great, sacred" for w in ['වහන්සේ', 'රහතන්', 'තෙරුන්', 'සඟ', 'වහන්සේගේ', 'ධර්ම', 'සංඝයා', 'බුදුන්']},
    **{w: "powerful, strong" for w in ['රජ', 'ප්රධාන', 'රජුන්', 'ඉසුරු', 'ඇමැති', 'රජානනි', 'රාජගුරු', 'අජාසත්' ,'නරනිඳු', 'ධර්මාශෝක', 'පතිරජ']},
    **{w: "big, large, massive" for w in ['ලක්ෂයක්', 'මහත්', 'සමූහයක්', 'මුහුද', 'කෛලාස']}
}

In [94]:
maha_collocates_df_filtered = maha_collocates_df[maha_collocates_df['collocate_word'].isin(maha_collocates)].copy()

In [95]:
col_pos = maha_collocates_df_filtered.columns.get_loc('collocate_word') + 1

In [96]:
meanings_column = maha_collocates_df_filtered['collocate_word'].map(meanings_map)

In [97]:
maha_collocates_df_filtered.insert(col_pos, 'Related Meaning(s)', meanings_column)

In [98]:
maha_collocates_df_filtered['collocate_word'] = pd.Categorical(maha_collocates_df_filtered['collocate_word'], categories=maha_collocates, ordered=True)
maha_collocates_df_filtered = maha_collocates_df_filtered.sort_values('collocate_word').reset_index(drop=True)

In [99]:
maha_collocates_df_filtered

,collocate_word,Related Meaning(s),13,14,15,16,18,19,20,total_count,century_span,longevity_score
0,වහන්සේ,"esteemed, great, sacred",1,6,0,0,0,2,2,11,4,44
1,රහතන්,"esteemed, great, sacred",0,2,0,0,0,1,4,7,3,21
2,තෙරුන්,"esteemed, great, sacred",1,3,0,0,0,1,0,5,3,15
3,සඟ,"esteemed, great, sacred",0,2,1,0,0,0,2,5,3,15
4,වහන්සේගේ,"esteemed, great, sacred",1,3,0,0,0,0,0,4,2,8
5,ධර්ම,"esteemed, great, sacred",1,1,0,0,0,0,1,3,3,9
6,සංඝයා,"esteemed, great, sacred",0,0,2,0,1,0,0,3,2,6
7,බුදුන්,"esteemed, great, sacred",2,0,0,0,0,0,1,3,2,6
8,රජ,"powerful, strong",0,1,0,0,0,0,8,9,2,18
9,ප්රධාන,"powerful, strong",1,0,0,0,0,0,1,2,2,4


In [100]:
maha_collocates_df_filtered.drop(columns=['total_count', 'century_span', 'longevity_score'], inplace=True)

In [101]:
maha_collocates_df_filtered

,collocate_word,Related Meaning(s),13,14,15,16,18,19,20
0,වහන්සේ,"esteemed, great, sacred",1,6,0,0,0,2,2
1,රහතන්,"esteemed, great, sacred",0,2,0,0,0,1,4
2,තෙරුන්,"esteemed, great, sacred",1,3,0,0,0,1,0
3,සඟ,"esteemed, great, sacred",0,2,1,0,0,0,2
4,වහන්සේගේ,"esteemed, great, sacred",1,3,0,0,0,0,0
5,ධර්ම,"esteemed, great, sacred",1,1,0,0,0,0,1
6,සංඝයා,"esteemed, great, sacred",0,0,2,0,1,0,0
7,බුදුන්,"esteemed, great, sacred",2,0,0,0,0,0,1
8,රජ,"powerful, strong",0,1,0,0,0,0,8
9,ප්රධාන,"powerful, strong",1,0,0,0,0,0,1


In [102]:
maha_collocates_df_filtered.to_latex('Tex_Files/maha_collocates_table.tex', index=False, encoding='utf-8-sig')

## 'සතර' Analysis

In [103]:
sathara_collocates_df = pd.read_csv('Collocates/සතර.csv')

In [104]:
sathara_collocates_df.head()

,collocate_word,13,14,15,16,18,19,20
0,අංගවන,1,0,0,0,0,0,0
1,අක්,2,0,0,0,0,0,0
2,අක්ෂර,0,0,0,0,0,1,0
3,අකුරෙව්,0,0,0,1,0,0,0
4,අග,0,0,0,1,0,0,0


In [105]:
sathara_collocates_df = sathara_collocates_df.replace(r'^\s*$', np.nan, regex=True).fillna(0)

In [106]:
sathara_collocates_df.isna().sum()

collocate_word    0
13                0
14                0
15                0
16                0
18                0
19                0
20                0
dtype: int64

In [107]:
cols_to_sum = ['13', '14', '15', '16', '18', '19', '20']
sathara_collocates_df[cols_to_sum] = sathara_collocates_df[cols_to_sum].apply(pd.to_numeric, errors='coerce')
sathara_collocates_df['total_count'] = sathara_collocates_df[cols_to_sum].sum(axis=1)

In [108]:
sathara_collocates_df.sort_values(by='total_count', ascending=False, inplace=True)

In [109]:
sathara_collocates_df.head()

,collocate_word,13,14,15,16,18,19,20,total_count
330,පාද,0,0,0,0,0,8,1,9
465,රූප,0,0,0,0,3,4,0,7
134,ගණ,5,0,0,0,0,1,0,6
190,තුන්වන,2,0,0,0,2,1,0,5
502,වැනි,2,0,0,0,0,0,2,4


In [110]:
sathara_collocates_df['century_span'] = (sathara_collocates_df[cols_to_sum] > 0).sum(axis=1)

In [111]:
sathara_collocates_df['longevity_score'] = sathara_collocates_df['total_count'] * sathara_collocates_df['century_span']

In [112]:
sathara_collocates_df.sort_values(by='longevity_score', ascending=False, inplace=True)

In [113]:
sathara_collocates_df.reset_index(drop=True, inplace=True)

In [114]:
sathara_collocates_df.head(200)

,collocate_word,13,14,15,16,18,19,20,total_count,century_span,longevity_score
0,පාද,0,0,0,0,0,8,1,9,2,18
1,තුන්වන,2,0,0,0,2,1,0,5,3,15
2,රූප,0,0,0,0,3,4,0,7,2,14
3,ගණ,5,0,0,0,0,1,0,6,2,12
4,වැනි,2,0,0,0,0,0,2,4,2,8
5,රාජ,0,0,0,0,0,1,3,4,2,8
6,යනාදි,2,0,0,0,0,2,0,4,2,8
7,ෂට්,0,0,1,0,3,0,0,4,2,8
8,ලක්ෂණ,1,0,0,0,2,0,0,3,2,6
9,ජංඝා,2,0,0,0,1,0,0,3,2,6


In [115]:
sathara_collocates_df.to_csv('Collocates/sathara_collocates_analysis.csv', index=False, encoding='utf-8-sig')

In [116]:
sathara_collocates = ["දක්ෂ", "පඬින" #Intelligent
                   "උගනීද", "ඉගෙන", "සිප්", "ගුරෙකු", "ඉගෙණ", "සිප්සතරා", "ශික්ෂායි", "විද්යාස්ථාන", #Learn, Educate
                   "ගණ", "තුන්", "පංච", "අටකින්ද", "පණසකින්ද", "සතරක්", #Value, Math
                   "ඩැහැගෙණ", #Thief
                   "අපාය", #Hell
                   "ඥාන", "නුවණ", "ඥානය", "දත්", "දැනගන්ට", "නැණින්", #Wisdom, Knowledge
                #    "දිශා" #Direction
                   ]

In [117]:
sathara_collocates_df[sathara_collocates_df['collocate_word'].isin(sathara_collocates)]

,collocate_word,13,14,15,16,18,19,20,total_count,century_span,longevity_score
3,ගණ,5,0,0,0,0,1,0,6,2,12
10,සතරක්,0,0,0,2,0,1,0,3,2,6
14,තුන්,1,0,0,0,0,0,1,2,2,4
16,අපාය,1,0,1,0,0,0,0,2,2,4
20,ඥාන,0,0,0,0,4,0,0,4,1,4
30,පංච,0,0,0,0,3,0,0,3,1,3
43,ගුරෙකු,0,0,0,0,0,0,3,3,1,3
68,සිප්,0,0,0,0,0,0,2,2,1,2
122,අටකින්ද,0,0,0,0,0,1,0,1,1,1
152,ඉගෙණ,0,0,0,0,0,1,0,1,1,1


In [118]:
meanings_map = {
    **{w: "Intelligent" for w in ["දක්ෂ", "පඬින"]},
    **{w: "Learn, Educate" for w in ["උගනීද", "ඉගෙන", "සිප්", "ගුරෙකු", "ඉගෙණ", "සිප්සතරා", "ශික්ෂායි", "විද්යාස්ථාන"]},
    **{w: "Value, Math" for w in ["ගණ", "තුන්", "පංච", "අටකින්ද", "පණසකින්ද", "සතරක්"]},
    **{w: "Thief" for w in ["ඩැහැගෙණ"]},
    **{w: "Hell" for w in ["අපාය"]},
    **{w: "Wisdom, Knowledge" for w in ["ඥාන", "නුවණ", "ඥානය", "දත්", "දැනගන්ට", "නැණින්"]}
}

In [119]:
sathara_collocates_df[sathara_collocates_df['collocate_word'].isin(["දක්ෂ", "පඬින"])].copy()

,collocate_word,13,14,15,16,18,19,20,total_count,century_span,longevity_score


In [120]:
sathara_collocates_df_filtered = sathara_collocates_df[sathara_collocates_df['collocate_word'].isin(sathara_collocates)].copy()

In [121]:
sathara_collocates_df_filtered

,collocate_word,13,14,15,16,18,19,20,total_count,century_span,longevity_score
3,ගණ,5,0,0,0,0,1,0,6,2,12
10,සතරක්,0,0,0,2,0,1,0,3,2,6
14,තුන්,1,0,0,0,0,0,1,2,2,4
16,අපාය,1,0,1,0,0,0,0,2,2,4
20,ඥාන,0,0,0,0,4,0,0,4,1,4
30,පංච,0,0,0,0,3,0,0,3,1,3
43,ගුරෙකු,0,0,0,0,0,0,3,3,1,3
68,සිප්,0,0,0,0,0,0,2,2,1,2
122,අටකින්ද,0,0,0,0,0,1,0,1,1,1
152,ඉගෙණ,0,0,0,0,0,1,0,1,1,1


In [122]:
col_pos = sathara_collocates_df_filtered.columns.get_loc('collocate_word') + 1

In [123]:
meanings_column = sathara_collocates_df_filtered['collocate_word'].map(meanings_map)

In [124]:
sathara_collocates_df_filtered.insert(col_pos, 'Related Meaning(s)', meanings_column)

In [125]:
sathara_collocates_df_filtered['collocate_word'] = pd.Categorical(sathara_collocates_df_filtered['collocate_word'], categories=sathara_collocates, ordered=True)
sathara_collocates_df_filtered = sathara_collocates_df_filtered.sort_values('collocate_word').reset_index(drop=True)

In [126]:
sathara_collocates_df_filtered

,collocate_word,Related Meaning(s),13,14,15,16,18,19,20,total_count,century_span,longevity_score
0,සිප්,"Learn, Educate",0,0,0,0,0,0,2,2,1,2
1,ගුරෙකු,"Learn, Educate",0,0,0,0,0,0,3,3,1,3
2,ඉගෙණ,"Learn, Educate",0,0,0,0,0,1,0,1,1,1
3,සිප්සතරා,"Learn, Educate",0,0,0,0,0,0,1,1,1,1
4,ශික්ෂායි,"Learn, Educate",1,0,0,0,0,0,0,1,1,1
5,විද්යාස්ථාන,"Learn, Educate",1,0,0,0,0,0,0,1,1,1
6,ගණ,"Value, Math",5,0,0,0,0,1,0,6,2,12
7,තුන්,"Value, Math",1,0,0,0,0,0,1,2,2,4
8,පංච,"Value, Math",0,0,0,0,3,0,0,3,1,3
9,අටකින්ද,"Value, Math",0,0,0,0,0,1,0,1,1,1


In [127]:
sathara_collocates_df_filtered.drop(columns=['total_count', 'century_span', 'longevity_score'], inplace=True)

In [128]:
sathara_collocates_df_filtered

,collocate_word,Related Meaning(s),13,14,15,16,18,19,20
0,සිප්,"Learn, Educate",0,0,0,0,0,0,2
1,ගුරෙකු,"Learn, Educate",0,0,0,0,0,0,3
2,ඉගෙණ,"Learn, Educate",0,0,0,0,0,1,0
3,සිප්සතරා,"Learn, Educate",0,0,0,0,0,0,1
4,ශික්ෂායි,"Learn, Educate",1,0,0,0,0,0,0
5,විද්යාස්ථාන,"Learn, Educate",1,0,0,0,0,0,0
6,ගණ,"Value, Math",5,0,0,0,0,1,0
7,තුන්,"Value, Math",1,0,0,0,0,0,1
8,පංච,"Value, Math",0,0,0,0,3,0,0
9,අටකින්ද,"Value, Math",0,0,0,0,0,1,0


In [129]:
sathara_collocates_df_filtered.to_latex('Tex_Files/sathara_collocates_table.tex', index=False, encoding='utf-8-sig')